# Set up

In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import random
import os
from pathlib import Path
import time
import gc
from tqdm.auto import tqdm

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

# Configuration

In [ ]:
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'
region_dir = base_dir / 'region_names'

weights_dir = base_dir / 'models_elasticnet_dnanexus/elasticnet_weights_scaling_law'
results_dir = base_dir / 'models_elasticnet_dnanexus/elasticnet_scaling_law_results'
predictions_dir = base_dir / 'models_elasticnet_dnanexus/elasticnet_predictions_scaling_law'
os.makedirs(weights_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)

rename = pd.read_csv(region_dir / 'col_renames_dnanexus.csv')
rename_dict = dict(zip(rename['datafield_code'], rename['datafield_name']))

targets = {
    'GF': ('GF', 'p20016_i2'),
    'PAL': ('PAL', 'p20197_i2'),
    'DSST': ('DSST', 'p23324_i2'),
    'TMT': ('TMT', 'p6350_i2'),
}

data_configs = {
    'demo': (None, ['p31', 'p21003_i2', 'p54_i2']),
    'MIND_avg': (region_dir / 'MIND_avg_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
    'CT': (region_dir / 'CT_regions_dnanexus.txt', ['p31', 'p21003_i2', 'p54_i2']),
    'FC25': (region_dir / 'FC25_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'FC100': (region_dir / 'FC100_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'MIND': (region_dir / 'MIND_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
}

sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']


# ElasticNet Analysis Function

In [ ]:
#inner parallelized
def elasticnet_analysis(X, y, continuous_vars, categorical_vars, weights_dir, predictions_dir, data_name, target_name, sample_size, n_splits=10):
    preprocessor = ColumnTransformer(transformers=[
        # scale continuous features
        ('num', StandardScaler(), continuous_vars),
        # one-hot encode assessment centre and sex
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_vars),
    ])

    # Cross-validation set-up
    outer_cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    outer_mae, outer_rmse, outer_r2, outer_r2_corr = [], [], [], []
    best_params_per_fold = []
    outer_coefs = [] # <-- Added list to collect coefficients per fold

    target_weights_dir = os.path.join(weights_dir, target_name)
    os.makedirs(target_weights_dir, exist_ok=True)
    target_predictions_dir = os.path.join(predictions_dir, target_name)
    os.makedirs(target_predictions_dir, exist_ok=True)
    preds_filename = f'ElasticNet_preds_{data_name}_{target_name}_{sample_size}.csv'
    preds_path = os.path.join(target_predictions_dir, preds_filename)

    cv_splits = tqdm(
        outer_cv.split(X, y), 
        total=n_splits, 
        desc=f"CV Folds ({target_name} on {data_name}, n={sample_size})", 
        leave=False
    )
    
    for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Inner CV using TransformedTargetRegressor for automatic y scaling
        pipe = TransformedTargetRegressor(
            regressor=make_pipeline(
                preprocessor,
                ElasticNetCV(
                    l1_ratio=[0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99,1.0],
                    alphas=np.logspace(-5,2,15),
                    cv=10, max_iter=40000, random_state=seed,
                    n_jobs=-1
                )
            ),
            transformer=StandardScaler()
        )

        # Pass y_train directly
        pipe.fit(X_train, y_train.values.reshape(-1, 1))
        y_pred = pipe.predict(X_test).ravel()

        # collect actual vs. predicted for this fold
        fold_df = pd.DataFrame({
            'fold': fold,
            'eid': y_test.index,
            'actual': y_test.values,
            'predicted': y_pred
        })
        fold_df.to_csv(preds_path, mode='a', header=(fold==1), index=False)
        del fold_df

        # metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        if np.std(y_pred) == 0:
             corr = 0.0
        else:
             corr = np.corrcoef(y_test, y_pred.squeeze())[0, 1]
        r2_corr = corr ** 2

        outer_mae.append(mae)
        outer_rmse.append(rmse)
        outer_r2.append(r2)
        outer_r2_corr.append(r2_corr)

        # store best α & l1_ratio and coefficients for this fold
        est = pipe.regressor_.named_steps['elasticnetcv']
        best_params_per_fold.append({'alpha': est.alpha_, 'l1_ratio': est.l1_ratio_})
        outer_coefs.append(est.coef_)

        del pipe
        gc.collect()

        print(f'  Fold {fold:02d} • MAE={mae:.3f} • RMSE={rmse:.3f} • R²={r2:.3f} • R²(corr)={r2_corr:.3f} '
              f'• α={est.alpha_:.4g} • l1_ratio={est.l1_ratio_:.2f}')
        

    # Aggregate results
    print(f'\n  Mean MAE    : {np.mean(outer_mae):.3f} ± {np.std(outer_mae):.3f}')
    print(f'  Mean RMSE   : {np.mean(outer_rmse):.3f} ± {np.std(outer_rmse):.3f}')
    print(f'  Mean R²     : {np.mean(outer_r2):.3f} ± {np.std(outer_r2):.3f}')
    print(f'  Mean R²(corr): {np.mean(outer_r2_corr):.3f} ± {np.std(outer_r2_corr):.3f}')

    # Final refit on all data
    final_pipe = TransformedTargetRegressor(
        regressor=make_pipeline(
            preprocessor,
            ElasticNetCV(
                l1_ratio=[0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99,1.0],
                alphas=np.logspace(-5,2,15),
                cv=10, max_iter=40000, random_state=seed,
                n_jobs=-1
            )
        ),
        transformer=StandardScaler()
    ).fit(X, y.values.reshape(-1, 1))

    # Access the natively found parameters from the final cross validation
    elasticnet_cv = final_pipe.regressor_.named_steps['elasticnetcv']
    final_alpha = elasticnet_cv.alpha_
    final_l1_ratio = elasticnet_cv.l1_ratio_

    # Feature names after preprocessing
    preprocessor_fitted = final_pipe.regressor_.named_steps['columntransformer']
    cat_features = list(preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_vars))
    all_feature_names = continuous_vars + cat_features

    # --- Process outer fold coefficients and identify surviving features ---
    outer_coefs = np.array(outer_coefs)  # Shape: (n_splits, n_features)
    mean_coef = np.mean(outer_coefs, axis=0)
    std_coef = np.std(outer_coefs, axis=0)
    se_coef = std_coef / np.sqrt(n_splits)
    
    # Calculate 95% Confidence Interval
    ci_lower = mean_coef - (1.96 * se_coef)
    ci_upper = mean_coef + (1.96 * se_coef)
    
    # A feature survives if its CI doesn't cross 0 (i.e. strictly positive or strictly negative)
    surviving_mask = (ci_lower > 0) | (ci_upper < 0)
    surviving_features = np.array(all_feature_names)[surviving_mask].tolist()
    num_surviving = len(surviving_features)

    # Save tracking to txt file
    surviving_filename = f'ElasticNet_surviving_{data_name}_{target_name}_{sample_size}.txt'
    with open(os.path.join(target_weights_dir, surviving_filename), 'w') as f:
        for feature in surviving_features:
            f.write(f"{feature}\n")

    # Save original weights df
    coefs_df = pd.DataFrame({
        'Feature': all_feature_names, 
        'Coefficient': elasticnet_cv.coef_,
        'CV_Mean': mean_coef,
        'CV_Std': std_coef
    })
    weights_filename = f'ElasticNet_weights_{data_name}_{target_name}_{sample_size}.csv'
    coefs_df.to_csv(os.path.join(target_weights_dir, weights_filename), index=False)
    # ----------------------------------------------------------------------------

    num_nonzero = int(np.sum(elasticnet_cv.coef_ != 0))

    del coefs_df
    del final_pipe
    del elasticnet_cv
    gc.collect()

    print(f'  Weights → {weights_filename}')
    print(f'  Predictions → {preds_filename}')
    print(f'  Surviving Features → {surviving_filename}')
    print(f'  Final params: α={final_alpha:.4g}, l1_ratio={final_l1_ratio:.3f}, nonzero={num_nonzero}, surviving={num_surviving}')

    return {
        'mean_mae':         np.mean(outer_mae),
        'std_mae':          np.std(outer_mae),
        'mean_rmse':        np.mean(outer_rmse),
        'std_rmse':         np.std(outer_rmse),
        'mean_r2':          np.mean(outer_r2),
        'std_r2':           np.std(outer_r2),
        'mean_r2_corr':     np.mean(outer_r2_corr),
        'std_r2_corr':      np.std(outer_r2_corr),
        'alpha':            final_alpha,
        'l1_ratio':         final_l1_ratio,
        'num_nonzero_coefs': num_nonzero,
        'num_surviving':    num_surviving
    }

# Scaling Law Training Loop

In [ ]:
for target_name, (test_key, score_col) in targets.items():
    print(f'\n{"="*60}\nTARGET: {target_name}\n{"="*60}')

    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    target_splits_dir = splits_dir / target_name

    df_full = pd.read_csv(data_file, index_col=0)

    for data_name, (regions_file, demographic_vars) in data_configs.items():
        print(f'\n--- {target_name} vs. {data_name} ---')

        if regions_file is not None:
            with open(regions_file, 'r') as f:
                brain_regions = [line.strip() for line in f]
        else:
            brain_regions = []

        all_vars = demographic_vars + brain_regions

        for sample_size in sample_sizes:
            if sample_size == 'all':
                eid_file = target_splits_dir / f'{target_name}_all_eids.txt'
            else:
                eid_file = target_splits_dir / f'{target_name}_eids_{sample_size}.txt'

            if not eid_file.exists():
                print(f'  Skipping n={sample_size}: EID file not found.')
                continue

            sample_eids = np.loadtxt(eid_file, dtype=int)

           # Filter to sample -- handles eid as either column or index
            if 'eid' in df_full.columns:
                df = df_full[df_full['eid'].isin(sample_eids)]
            else:
                df = df_full[df_full.index.isin(sample_eids)]

            actual_n = len(df)
            print(f'\n[{target_name} | {data_name} | n={sample_size} ({actual_n} rows)]')

            X = df[all_vars].rename(columns=rename_dict)
            y = df[score_col]

            categorical_vars = ['sex', 'assessment_centre']
            continuous_vars  = [c for c in X.columns if c not in categorical_vars]

            start_time = time.time()

            metrics = elasticnet_analysis(
                X, y, continuous_vars, categorical_vars,
                weights_dir, predictions_dir, data_name, 
                target_name, sample_size
            )

            end_time = time.time()
            elapsed = end_time - start_time
            print(f'  Time taken for config [{data_name}, n={sample_size}]: {elapsed:.2f} seconds')

            results_file = results_dir / f'scaling_law_results_{target_name}.csv'
            row_df = pd.DataFrame([{
                'target_name': target_name,
                'data_name':   data_name,
                'sample_size': sample_size,
                'actual_n':    actual_n,
                **metrics,
                'elapsed_time_sec': elapsed
            }])
            row_df.to_csv(str(results_file), mode='a', header=not results_file.exists(), index=False)

            # Explicit memory cleanup
            del df, X, y, metrics
            gc.collect()

    print(f'\nResults saved → {results_file}')